# WRF U-Net Downscaling: Preprocessing, Training, Testing, and Inference

This notebook prepares WRF fields for a small U-Net, trains the model, tests it, and saves a simple prediction file.

## 1. Learning Goals

By the end of this notebook, you should be able to:

- compare coarse and fine WRF variables before training
- see how each preprocessing step changes the data
- save lightweight processed train/test NPZ files
- train a residual U-Net and visualize one prediction

## 2. Set Up Paths

Run this notebook from inside the `dl_downscaling_teaching` directory so the local imports resolve correctly.

In [ ]:
import os

import config as cfg

# Create the output folder before saving checkpoints, plots, or predictions.
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Print the main paths so students know where files are read and written.
print("Teaching directory:    ", cfg.PROJECT_DIR)
print("Data directory:        ", cfg.DATA_DIR)
print("Output directory:      ", cfg.OUTPUT_DIR)

## 3. Import Libraries

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision.transforms import InterpolationMode, Normalize, Resize
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from tqdm.auto import tqdm

from Network import UNet
from plot_utils import plot_loss_curve, plot_weather_grid, plot_weather_rows

# Use GPU when Colab provides one; otherwise use CPU.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

# Keep variable lists in one place so later cells use the same channel order.
WEATHER_VARIABLES = list(cfg.IN_OUT_PARAMS)
STATIC_VARIABLES = list(cfg.CONSTANT_PARAMS)

print("Using device:", DEVICE)
print("Weather variables:", WEATHER_VARIABLES)
print("Static variables: ", STATIC_VARIABLES)

## 4. Data Helpers

These small helpers load the source NPZ files and reconstruct fine fields after prediction.

In [ ]:
def load_variable_stats(stats, variables):
    # Keep means and standard deviations in the same order as model channels.
    means = []
    stds = []
    for variable in variables:
        means.append(stats["mean"][variable])
        stds.append(stats["std"][variable])
    return means, stds


def raw_variable_name(variable_name):
    # The raw NPZ uses ACC_6H_PRECIP before the notebook applies the log transform.
    if variable_name == "LN_ACC_6H_PRECIP":
        return "ACC_6H_PRECIP"
    return variable_name


def read_weather_channel(data, variable_names, variable_name):
    # Select one channel from the raw data array by variable name.
    source_name = raw_variable_name(variable_name)
    variable_index = list(variable_names).index(source_name)
    channel = data[:, variable_index]

    # Rainfall is transformed to log(precipitation + 0.001).
    if variable_name == "LN_ACC_6H_PRECIP":
        channel = np.maximum(channel, 0.0)
        channel = np.log(channel + 0.001)

    return channel.astype(np.float32)


def load_weather_file(path):
    # Raw files store all weather channels in one array named data.
    with np.load(path, allow_pickle=False) as npz_file:
        raw_data = npz_file["data"]
        variable_names = npz_file["variable_names"]
        channels = []
        for name in WEATHER_VARIABLES:
            channel = read_weather_channel(raw_data, variable_names, name)
            channels.append(channel)

        # Final shape is time, channel, y, x.
        data = np.stack(channels, axis=1)
        time = npz_file["time"]
        lat = npz_file["lat"]
        lon = npz_file["lon"]
    return data, time, lat, lon


def residual_to_fine(residual_norm, coarse, mean, std):
    # Convert normalized residuals back to physical units, then add coarse fields.
    mean = torch.tensor(mean, dtype=residual_norm.dtype).view(1, -1, 1, 1)
    std = torch.tensor(std, dtype=residual_norm.dtype).view(1, -1, 1, 1)
    mean = mean.to(residual_norm.device)
    std = std.to(residual_norm.device)
    residual = residual_norm * std + mean
    return coarse + residual

## 5. Demo: Plot All Coarse And Fine Variables

The first plot shows every weather variable for one timestep before any model preprocessing.

In [ ]:
# Pick one timestep for all demonstration plots.
SAMPLE_TIME_INDEX = 0
SAMPLE_VARIABLE = "T2"
variable_index = WEATHER_VARIABLES.index(SAMPLE_VARIABLE)

# Load the original train coarse and fine data before preprocessing.
coarse_raw, coarse_time, coarse_lat, coarse_lon = load_weather_file(cfg.TRAIN_COARSE_RAW_NPZ_PATH)
fine_raw, fine_time, fine_lat, fine_lon = load_weather_file(cfg.TRAIN_FINE_RAW_NPZ_PATH)

# Build one plot panel for every coarse weather variable.
items = []
for i in range(len(WEATHER_VARIABLES)):
    name = WEATHER_VARIABLES[i]
    item = {
        "field": coarse_raw[SAMPLE_TIME_INDEX, i],
        "variable": name,
        "title": "Coarse " + name,
        "lat": coarse_lat,
        "lon": coarse_lon,
    }
    items.append(item)

# Add one plot panel for every fine weather variable.
for i in range(len(WEATHER_VARIABLES)):
    name = WEATHER_VARIABLES[i]
    item = {
        "field": fine_raw[SAMPLE_TIME_INDEX, i],
        "variable": name,
        "title": "Fine " + name,
        "lat": fine_lat,
        "lon": fine_lon,
    }
    items.append(item)

# Plot all panels in two rows: coarse first, fine second.
fig = plot_weather_grid(items, ncols=len(WEATHER_VARIABLES), figsize_per_panel=(3.0, 2.8))
display(fig)
plt.close(fig)

print("Sample time:", coarse_time[SAMPLE_TIME_INDEX])
print("Raw coarse shape:", coarse_raw.shape)
print("Raw fine shape:  ", fine_raw.shape)

## 6. Choose Raw Files And Normalization Statistics

In [ ]:
# Load channel-wise normalization values from config.py.
raw_mean, raw_std = load_variable_stats(cfg.RAW_STATS, WEATHER_VARIABLES)
residual_mean, residual_std = load_variable_stats(cfg.RESIDUAL_STATS, WEATHER_VARIABLES)

print("Train coarse file:", cfg.TRAIN_COARSE_RAW_NPZ_PATH)
print("Train fine file:  ", cfg.TRAIN_FINE_RAW_NPZ_PATH)
print("Test coarse file: ", cfg.TEST_COARSE_RAW_NPZ_PATH)
print("Test fine file:   ", cfg.TEST_FINE_RAW_NPZ_PATH)

# Round long decimals only for display.
rounded_raw_mean = []
for value in raw_mean:
    rounded_raw_mean.append(round(value, 3))

rounded_residual_std = []
for value in residual_std:
    rounded_residual_std.append(round(value, 3))

print("Raw means:     ", rounded_raw_mean)
print("Residual stds: ", rounded_residual_std)
print("Epochs:        ", cfg.NUM_EPOCHS)
print("Batch size:    ", cfg.BATCH_SIZE)
print("Learning rate: ", cfg.LEARNING_RATE)

## 7. Preprocessing Step 1: Match Coarse And Fine Grids

Both domains are resized to the 48 x 48 model grid before residuals are computed.

In [ ]:
# Convert NumPy arrays into PyTorch tensors before resizing.
coarse_demo = torch.tensor(coarse_raw, dtype=torch.float32)
fine_demo = torch.tensor(fine_raw, dtype=torch.float32)

# Use nearest-neighbor resizing for the coarse grid.
coarse_resize = Resize(
    cfg.HIGH_RES_SHP,
    interpolation=InterpolationMode.NEAREST,
    antialias=False,
)
coarse_demo = coarse_resize(coarse_demo)

# Use bilinear resizing for the fine grid.
fine_resize = Resize(
    cfg.HIGH_RES_SHP,
    interpolation=InterpolationMode.BILINEAR,
    antialias=False,
)
fine_demo = fine_resize(fine_demo)

# Compare one variable before and after grid matching.
items = [
    {"field": coarse_raw[SAMPLE_TIME_INDEX, variable_index], "variable": SAMPLE_VARIABLE, "title": "Raw coarse", "lat": coarse_lat, "lon": coarse_lon},
    {"field": fine_raw[SAMPLE_TIME_INDEX, variable_index], "variable": SAMPLE_VARIABLE, "title": "Raw fine", "lat": fine_lat, "lon": fine_lon},
    {"field": coarse_demo[SAMPLE_TIME_INDEX, variable_index], "variable": SAMPLE_VARIABLE, "title": "Matched coarse"},
    {"field": fine_demo[SAMPLE_TIME_INDEX, variable_index], "variable": SAMPLE_VARIABLE, "title": "Matched fine"},
]
fig = plot_weather_grid(items, ncols=4)
display(fig)
plt.close(fig)

print("Matched coarse shape:", tuple(coarse_demo.shape))
print("Matched fine shape:  ", tuple(fine_demo.shape))

## 8. Preprocessing Step 2: Transform Precipitation

Accumulated precipitation is changed to `log(precipitation + 0.001)` so small and large rainfall values are easier for the model to learn together.

In [ ]:
# Open the raw train coarse file to show the rainfall transform directly.
with np.load(cfg.TRAIN_COARSE_RAW_NPZ_PATH, allow_pickle=False) as coarse_npz:
    variable_names = list(coarse_npz["variable_names"])
    precip_index = variable_names.index("ACC_6H_PRECIP")
    raw_precip = np.maximum(coarse_npz["data"][SAMPLE_TIME_INDEX, precip_index], 0.0)
    log_precip = np.log(raw_precip + 0.001)

# Plot raw rainfall beside its log-transformed version.
items = [
    {
        "field": raw_precip,
        "variable": "LN_ACC_6H_PRECIP",
        "title": "Actual precipitation",
        "lat": coarse_lat,
        "lon": coarse_lon,
        "colorbar_label": "mm",
        "precip_is_log": False,
    },
    {
        "field": log_precip,
        "variable": "LOG_PRECIP",
        "title": "Log precipitation",
        "lat": coarse_lat,
        "lon": coarse_lon,
        "colorbar_label": "log(mm + 0.001)",
    },
]
fig = plot_weather_grid(items, ncols=2)
display(fig)
plt.close(fig)

print("One grid cell before transform:", round(float(raw_precip[0, 0]), 3))
print("One grid cell after transform: ", round(float(log_precip[0, 0]), 3))

## 9. Preprocessing Step 3: Build Static Channels

Static channels give the model fixed information about the grid, such as land/sea mask and terrain height.

In [ ]:
# Static fields do not change with time, so load them once.
static_channels = []
with np.load(cfg.STATIC_NPZ_PATH, allow_pickle=False) as static_npz:
    for variable in STATIC_VARIABLES:
        channel = torch.tensor(static_npz[variable], dtype=torch.float32)[None, None]

        # Land-sea mask is categorical, so resize it with nearest neighbor.
        if variable == "LSM":
            static_resize = Resize(
                cfg.HIGH_RES_SHP,
                interpolation=InterpolationMode.NEAREST,
                antialias=False,
            )
        else:
            # Terrain height is continuous, so resize it with bilinear interpolation.
            static_resize = Resize(
                cfg.HIGH_RES_SHP,
                interpolation=InterpolationMode.BILINEAR,
                antialias=False,
            )

        channel = static_resize(channel)[0, 0]

        # Terrain height is standardized so its scale is easier for the model.
        if variable != "LSM":
            channel = (channel - channel.mean()) / (channel.std() + 1e-6)
        static_channels.append(channel)

static_demo = torch.stack(static_channels, dim=0)

# Plot the static channels used by the model.
items = []
for i in range(len(STATIC_VARIABLES)):
    name = STATIC_VARIABLES[i]
    item = {"field": static_demo[i], "variable": name, "title": name}
    items.append(item)
fig = plot_weather_grid(items, ncols=len(STATIC_VARIABLES))
display(fig)
plt.close(fig)

print("Static channel shape:", tuple(static_demo.shape))

## 10. Preprocessing Step 4: Build Residual Targets

The U-Net learns the difference between fine and coarse fields instead of predicting the full fine field directly.

In [ ]:
# The model learns fine minus coarse instead of the whole fine field.
residual_demo = fine_demo - coarse_demo

# Show the matched coarse field, residual target, and matched fine field.
items = [
    {"field": coarse_demo[SAMPLE_TIME_INDEX, variable_index], "variable": SAMPLE_VARIABLE, "title": "Matched coarse"},
    {"field": residual_demo[SAMPLE_TIME_INDEX, variable_index], "variable": SAMPLE_VARIABLE, "title": "Fine - coarse residual"},
    {"field": fine_demo[SAMPLE_TIME_INDEX, variable_index], "variable": SAMPLE_VARIABLE, "title": "Matched fine"},
]
fig = plot_weather_grid(items, ncols=3)
display(fig)
plt.close(fig)

# Print one center-cell residual value as a numeric example.
row = cfg.HIGH_RES_SHP[0] // 2
col = cfg.HIGH_RES_SHP[1] // 2
print("Center grid-cell residual:", round(float(residual_demo[SAMPLE_TIME_INDEX, variable_index, row, col]), 3))

## 11. Preprocessing Step 5: Normalize And Build Model Arrays

Weather inputs use raw-field statistics. Targets use residual statistics. The final model input is normalized weather plus static channels.

In [ ]:
# Repeat static channels so every timestep receives the same static information.
static_channels = static_demo.expand(coarse_demo.shape[0], -1, -1, -1)

# Normalize the coarse weather fields before giving them to the model.
input_normalizer = Normalize(mean=raw_mean, std=raw_std)
weather_inputs_demo = input_normalizer(coarse_demo)

# Normalize residual targets so all output channels have comparable scale.
target_normalizer = Normalize(mean=residual_mean, std=residual_std)
targets_demo = target_normalizer(residual_demo)

# Final model input is weather channels plus static channels.
inputs_demo = torch.cat([weather_inputs_demo, static_channels], dim=1)

# Plot physical and normalized values side by side.
items = [
    {"field": coarse_demo[SAMPLE_TIME_INDEX, variable_index], "variable": SAMPLE_VARIABLE, "title": "Physical coarse"},
    {"field": inputs_demo[SAMPLE_TIME_INDEX, variable_index], "variable": "NORMALIZED", "title": "Normalized coarse", "colorbar_label": "standardized value"},
    {"field": residual_demo[SAMPLE_TIME_INDEX, variable_index], "variable": SAMPLE_VARIABLE, "title": "Physical residual"},
    {"field": targets_demo[SAMPLE_TIME_INDEX, variable_index], "variable": "NORMALIZED", "title": "Normalized residual", "colorbar_label": "standardized value"},
]
fig = plot_weather_grid(items, ncols=4)
display(fig)
plt.close(fig)

print("Model inputs: ", tuple(inputs_demo.shape))
print("Model targets:", tuple(targets_demo.shape))

## 12. Save Processed Train And Test Files

Each processed file stores readable arrays named `inputs`, `targets`, `coarse`, `fine`, `time`, `lat`, and `lon`. Static-variable names and normalization constants stay in `config.py`.

In [ ]:
# Load the predefined raw training pair.
train_coarse_raw, train_time, train_coarse_lat, train_coarse_lon = load_weather_file(cfg.TRAIN_COARSE_RAW_NPZ_PATH)
train_fine_raw, train_fine_time, train_fine_lat, train_fine_lon = load_weather_file(cfg.TRAIN_FINE_RAW_NPZ_PATH)

# Convert and resize the training pair directly.
train_coarse = torch.tensor(train_coarse_raw, dtype=torch.float32)
train_fine = torch.tensor(train_fine_raw, dtype=torch.float32)
train_coarse = coarse_resize(train_coarse)
train_fine = fine_resize(train_fine)

# Build model-ready training arrays directly.
train_static = static_demo.expand(train_coarse.shape[0], -1, -1, -1)
train_weather_inputs = input_normalizer(train_coarse)
train_residual = train_fine - train_coarse
train_targets = target_normalizer(train_residual)
train_inputs = torch.cat([train_weather_inputs, train_static], dim=1)

# The saved fields are on the resized 48 x 48 teaching grid.
lat = np.linspace(cfg.DL_DOMAIN["max_lat"], cfg.DL_DOMAIN["min_lat"], cfg.HIGH_RES_SHP[0])
lon = np.linspace(cfg.DL_DOMAIN["min_lon"], cfg.DL_DOMAIN["max_lon"], cfg.HIGH_RES_SHP[1])

# Save the processed training arrays.
np.savez_compressed(
    cfg.TRAIN_PROCESSED_NPZ_PATH,
    inputs=train_inputs.numpy(),
    targets=train_targets.numpy(),
    coarse=train_coarse.numpy(),
    fine=train_fine.numpy(),
    time=train_time,
    lat=lat,
    lon=lon,
)

# Load the predefined raw testing pair.
test_coarse_raw, test_time, test_coarse_lat, test_coarse_lon = load_weather_file(cfg.TEST_COARSE_RAW_NPZ_PATH)
test_fine_raw, test_fine_time, test_fine_lat, test_fine_lon = load_weather_file(cfg.TEST_FINE_RAW_NPZ_PATH)

# Convert and resize the testing pair directly.
test_coarse = torch.tensor(test_coarse_raw, dtype=torch.float32)
test_fine = torch.tensor(test_fine_raw, dtype=torch.float32)
test_coarse = coarse_resize(test_coarse)
test_fine = fine_resize(test_fine)

# Build model-ready testing arrays directly.
test_static = static_demo.expand(test_coarse.shape[0], -1, -1, -1)
test_weather_inputs = input_normalizer(test_coarse)
test_residual = test_fine - test_coarse
test_targets = target_normalizer(test_residual)
test_inputs = torch.cat([test_weather_inputs, test_static], dim=1)

# Save the processed testing arrays.
np.savez_compressed(
    cfg.TEST_PROCESSED_NPZ_PATH,
    inputs=test_inputs.numpy(),
    targets=test_targets.numpy(),
    coarse=test_coarse.numpy(),
    fine=test_fine.numpy(),
    time=test_time,
    lat=lat,
    lon=lon,
)

print("Saved train file:", cfg.TRAIN_PROCESSED_NPZ_PATH)
print("Train shapes:", train_inputs.shape, train_targets.shape, train_coarse.shape, train_fine.shape)
print("Saved test file: ", cfg.TEST_PROCESSED_NPZ_PATH)
print("Test shapes: ", test_inputs.shape, test_targets.shape, test_coarse.shape, test_fine.shape)

## 13. Inspect One Processed File

In [ ]:
# Open the processed train file and inspect each saved array.
with np.load(cfg.TRAIN_PROCESSED_NPZ_PATH, allow_pickle=False) as prepared:
    print("Keys:", prepared.files)
    print("Inputs: ", prepared["inputs"].shape, prepared["inputs"].dtype)
    print("Targets:", prepared["targets"].shape, prepared["targets"].dtype)
    print("Coarse: ", prepared["coarse"].shape, prepared["coarse"].dtype)
    print("Fine:   ", prepared["fine"].shape, prepared["fine"].dtype)
    print("Time:   ", prepared["time"].shape, prepared["time"].dtype)
    print("Lat:    ", prepared["lat"].shape, prepared["lat"].dtype)
    print("Lon:    ", prepared["lon"].shape, prepared["lon"].dtype)

## 14. Lightweight Dataset For Processed NPZ Files

In [ ]:
class ProcessedWRFDataset(torch.utils.data.Dataset):
    def __init__(self, path):
        # Load named arrays from the processed NPZ file.
        with np.load(path, allow_pickle=False) as npz_file:
            self.inputs = torch.from_numpy(npz_file["inputs"])
            self.targets = torch.from_numpy(npz_file["targets"])
            self.coarse = torch.from_numpy(npz_file["coarse"])
            self.fine = torch.from_numpy(npz_file["fine"])
            self.time = npz_file["time"]
            self.lat = npz_file["lat"]
            self.lon = npz_file["lon"]

    def __len__(self):
        # Number of samples equals the number of timesteps.
        return self.inputs.shape[0]

    def __getitem__(self, index):
        # Return one timestep as a dictionary for the training loop.
        return {
            "inputs": self.inputs[index],
            "targets": self.targets[index],
            "coarse": self.coarse[index],
            "fine": self.fine[index],
        }

## 15. Create Train And Test DataLoaders

In [ ]:
# Load train and test datasets from the two processed NPZ files.
train_dataset = ProcessedWRFDataset(cfg.TRAIN_PROCESSED_NPZ_PATH)
test_dataset = ProcessedWRFDataset(cfg.TEST_PROCESSED_NPZ_PATH)

# DataLoaders handle batching during training and testing.
train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0)

# Inspect one sample so students can see the tensor shapes.
first_sample = train_dataset[0]
print("Train samples:", len(train_dataset))
print("Test samples: ", len(test_dataset))
print("Input tensor shape: ", first_sample["inputs"].shape)
print("Target tensor shape:", first_sample["targets"].shape)

## 16. Build The Model And Loss

In [ ]:
# The input includes weather channels plus static channels.
num_input_channels = len(WEATHER_VARIABLES) + len(STATIC_VARIABLES)
num_output_channels = len(WEATHER_VARIABLES)

# Build a plain U-Net: no class labels and no time input.
model = UNet(
    img_resolution=cfg.HIGH_RES_SHP,
    in_channels=num_input_channels,
    out_channels=num_output_channels,
    label_dim=0,
    use_diffuse=False,
    **cfg.MODEL_KWARGS
)
model = model.to(DEVICE)

# Mean squared error compares predicted residuals with target residuals.
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE)

print("Input channels: ", num_input_channels)
print("Output channels:", num_output_channels)
print("Model is ready on", DEVICE)

## 17. Train And Test Functions

In [ ]:
def run_train_epoch(model, dataloader, optimizer):
    # Training mode enables gradient updates.
    model.train()
    losses = []

    for batch in tqdm(dataloader, desc="Train", leave=False):
        inputs = batch["inputs"].to(DEVICE)
        targets = batch["targets"].to(DEVICE)

        # Predict normalized residuals from normalized coarse inputs.
        prediction = model(inputs)
        loss = loss_fn(prediction, targets)

        # Standard PyTorch training step.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses))


def run_test_epoch(model, dataloader):
    # Evaluation mode avoids training-only behavior.
    model.eval()
    losses = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Test", leave=False):
            inputs = batch["inputs"].to(DEVICE)
            targets = batch["targets"].to(DEVICE)

            prediction = model(inputs)
            loss = loss_fn(prediction, targets)
            losses.append(loss.item())

    return float(np.mean(losses))

## 18. Run Training

In [ ]:
# Keep loss history so we can plot progress after each epoch.
train_losses = []
test_losses = []
best_test_loss = float("inf")

for epoch in range(1, cfg.NUM_EPOCHS + 1):
    train_loss = run_train_epoch(model, train_loader, optimizer)
    test_loss = run_test_epoch(model, test_loader)

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    # Save the best model according to test loss.
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        torch.save(model.state_dict(), cfg.BEST_CHECKPOINT_PATH)

    # Refresh the loss plot inside the notebook output.
    fig = plot_loss_curve(train_losses, test_losses, num_epochs=cfg.NUM_EPOCHS, title="Training Curve")
    fig.savefig(cfg.LOSS_CURVE_PATH, dpi=300, bbox_inches="tight")
    clear_output(wait=True)
    display(fig)
    plt.close(fig)

    # Print the full loss table after refreshing the plot.
    for i in range(len(train_losses)):
        completed_epoch = i + 1
        train_value = train_losses[i]
        test_value = test_losses[i]
        print(f"Epoch {completed_epoch} train loss = {train_value:.4f} test loss = {test_value:.4f}")

print("Best checkpoint saved to: ", cfg.BEST_CHECKPOINT_PATH)

## 19. Visualize One Test Prediction

In [ ]:
def plot_prediction_example(model, dataset, index=0):
    # Pick one sample and predict its normalized residual.
    model.eval()
    sample = dataset[index]

    inputs = sample["inputs"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        residual_prediction = model(inputs).cpu()

    # Reconstruct fine fields using the helper from the data section.
    predicted_fine = residual_to_fine(
        residual_prediction,
        sample["coarse"].unsqueeze(0),
        residual_mean,
        residual_std,
    )[0]

    # Plot coarse input, model prediction, and fine truth.
    rows = [
        ("Coarse", sample["coarse"]),
        ("Predicted", predicted_fine),
        ("Fine", sample["fine"]),
    ]
    return plot_weather_rows(rows, WEATHER_VARIABLES)


example_index = min(10, len(test_dataset) - 1)
fig = plot_prediction_example(model, test_dataset, index=example_index)
display(fig)
plt.close(fig)

## 20. Reload The Best Checkpoint For Inference

In [ ]:
# Reload the best checkpoint before running inference.
model.load_state_dict(torch.load(cfg.BEST_CHECKPOINT_PATH, map_location=DEVICE))
model.eval()

print("Loaded checkpoint for inference:", cfg.BEST_CHECKPOINT_PATH)

## 21. Predict The Test Dataset

In [ ]:
def predict_dataset(model, dataset, batch_size):
    # Use a non-shuffled DataLoader so outputs stay in dataset order.
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    coarse_batches = []
    fine_batches = []
    predicted_batches = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Inference", leave=False):
            inputs = batch["inputs"].to(DEVICE)

            # Predict normalized residuals, then reconstruct physical fine fields.
            residual_prediction = model(inputs).cpu()
            predicted_fine = residual_to_fine(
                residual_prediction,
                batch["coarse"],
                residual_mean,
                residual_std,
            )

            coarse_batches.append(batch["coarse"])
            fine_batches.append(batch["fine"])
            predicted_batches.append(predicted_fine)

    # Join all mini-batches into full arrays.
    coarse = torch.cat(coarse_batches, dim=0).numpy()
    fine = torch.cat(fine_batches, dim=0).numpy()
    predicted = torch.cat(predicted_batches, dim=0).numpy()
    return coarse, fine, predicted


# Run inference on the test dataset and save predictions.
coarse_output, fine_output, predicted_output = predict_dataset(model, test_dataset, cfg.BATCH_SIZE)

# Save coarse, predicted, and fine fields as three readable NPZ files.
COARSE_TEST_PATH = f"{cfg.OUTPUT_DIR}/wrf_test_coarse.npz"
PREDICTED_TEST_PATH = f"{cfg.OUTPUT_DIR}/wrf_test_predicted.npz"
FINE_TEST_PATH = f"{cfg.OUTPUT_DIR}/wrf_test_fine.npz"

np.savez_compressed(
    COARSE_TEST_PATH,
    data=coarse_output,
    time=test_dataset.time,
    lat=test_dataset.lat,
    lon=test_dataset.lon,
)
np.savez_compressed(
    PREDICTED_TEST_PATH,
    data=predicted_output,
    time=test_dataset.time,
    lat=test_dataset.lat,
    lon=test_dataset.lon,
)
np.savez_compressed(
    FINE_TEST_PATH,
    data=fine_output,
    time=test_dataset.time,
    lat=test_dataset.lat,
    lon=test_dataset.lon,
)

print("Coarse shape:   ", coarse_output.shape)
print("Fine shape:     ", fine_output.shape)
print("Predicted shape:", predicted_output.shape)
print("Saved coarse file:   ", COARSE_TEST_PATH)
print("Saved predicted file:", PREDICTED_TEST_PATH)
print("Saved fine file:     ", FINE_TEST_PATH)

## 22. Visualize One Saved Prediction

In [ ]:
# Display one saved-style prediction example.
rows = [
    ("Coarse", coarse_output[0]),
    ("Predicted", predicted_output[0]),
    ("Fine", fine_output[0]),
]

fig = plot_weather_rows(rows, WEATHER_VARIABLES)
display(fig)
plt.close(fig)